In [1]:
# Forecast configuration
HORIZON = 28
TRAIN_WINDOW_DAYS = 365

# Last day with observed sales
DATA_END_D = 1913

# Historical 28-day backtest
VALID_END_D = DATA_END_D
VALID_START_D = VALID_END_D - HORIZON + 1
TRAIN_END_D = VALID_START_D - 1
TRAIN_START_D = TRAIN_END_D - TRAIN_WINDOW_DAYS + 1

# Actual future forecast window
FORECAST_START_D = DATA_END_D + 1
FORECAST_END_D = DATA_END_D + HORIZON

# Target
target_col = "sales"

# Preserve the feature set from your previous baseline
feature_cols = [
    # Product information
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",

    # Time features
    "year",
    "month",
    "day_of_week",
    "is_weekend",

    # Historical demand
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_28",
    "rolling_mean_7",
    "rolling_mean_28",

    # Price features
    "sell_price",
    "price_available",
    "price_change",
    "price_changed",

    # Event features
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "is_event"
]

categorical_features = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

history_cols = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_28",
    "rolling_mean_7",
    "rolling_mean_28"
]

print(f"Training dates: d_{TRAIN_START_D} to d_{TRAIN_END_D}")
print(f"Validation dates: d_{VALID_START_D} to d_{VALID_END_D}")
print(
    f"Final forecast dates: "
    f"d_{FORECAST_START_D} to d_{FORECAST_END_D}"
)

print(f"\nExpected training rows: {30_490 * TRAIN_WINDOW_DAYS:,}")
print(f"Expected validation rows: {30_490 * HORIZON:,}")
print(f"Expected future predictions: {30_490 * HORIZON:,}")
print(f"Number of features: {len(feature_cols)}")

Training dates: d_1521 to d_1885
Validation dates: d_1886 to d_1913
Final forecast dates: d_1914 to d_1941

Expected training rows: 11,128,850
Expected validation rows: 853,720
Expected future predictions: 853,720
Number of features: 23


In [2]:
#Set the forecasting dates
import os
import glob

# Location of our corrected feature-engineered Parquet files
feature_folder = "../data/processed/feature_engineered_chunks"

feature_files = sorted(
    glob.glob(os.path.join(feature_folder, "*.parquet"))
)

# Forecast configuration
DATA_END_D = 1913
HORIZON = 28
TRAIN_WINDOW_DAYS = 365

# Historical validation: simulate a 28-day forecast
VALID_START_D = DATA_END_D - HORIZON + 1  # 1886
VALID_END_D = DATA_END_D                  # 1913

# Training period ends immediately before validation
TRAIN_END_D = VALID_START_D - 1           # 1885
TRAIN_START_D = TRAIN_END_D - TRAIN_WINDOW_DAYS + 1  # 1521

# Dates we will forecast after final training
FORECAST_START_D = DATA_END_D + 1         # 1914
FORECAST_END_D = DATA_END_D + HORIZON     # 1941

# Find which source chunks overlap each period
def get_chunk_bounds(file_path):
    filename = os.path.splitext(
        os.path.basename(file_path)
    )[0]

    chunk_start = int(filename.split("_")[-1])

    return chunk_start + 1, min(
        chunk_start + 200, DATA_END_D
    )

train_source_files = []
valid_source_files = []

for file in feature_files:
    chunk_start, chunk_end = get_chunk_bounds(file)

    if chunk_start <= TRAIN_END_D and chunk_end >= TRAIN_START_D:
        train_source_files.append(file)

    if chunk_start <= VALID_END_D and chunk_end >= VALID_START_D:
        valid_source_files.append(file)

print("Feature files found:", len(feature_files))
print("Training source files:", len(train_source_files))
print("Validation source files:", len(valid_source_files))

print(f"\nTraining days: d_{TRAIN_START_D} to d_{TRAIN_END_D}")
print(f"Validation days: d_{VALID_START_D} to d_{VALID_END_D}")
print(f"Future forecast: d_{FORECAST_START_D} to d_{FORECAST_END_D}")

print(f"\nExpected training rows: {30_490 * TRAIN_WINDOW_DAYS:,}")
print(f"Expected validation rows: {30_490 * HORIZON:,}")

Feature files found: 100
Training source files: 30
Validation source files: 10

Training days: d_1521 to d_1885
Validation days: d_1886 to d_1913
Future forecast: d_1914 to d_1941

Expected training rows: 11,128,850
Expected validation rows: 853,720


In [3]:
#Inspect one training chunk
import pandas as pd
import numpy as np
import gc

# Target we want to predict
target_col = "sales"

# Keep the same features as your previous baseline
feature_cols = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "year", "month", "day_of_week", "is_weekend",
    "sales_lag_1", "sales_lag_7", "sales_lag_28",
    "rolling_mean_7", "rolling_mean_28",
    "sell_price", "price_available", "price_change", "price_changed",
    "event_name_1", "event_type_1",
    "event_name_2", "event_type_2", "is_event"
]

categorical_features = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "event_name_1", "event_type_1",
    "event_name_2", "event_type_2"
]

history_cols = [
    "sales_lag_1", "sales_lag_7", "sales_lag_28",
    "rolling_mean_7", "rolling_mean_28"
]

# Read only the columns required for training
required_columns = list(
    dict.fromkeys(feature_cols + [target_col, "d"])
)

test_file = train_source_files[0]

sample_train = pd.read_parquet(
    test_file,
    columns=required_columns
)

# Keep dates that overlap the training period
sample_train = sample_train[
    sample_train["d"].between(TRAIN_START_D, TRAIN_END_D)
].copy()

print("Test file:", os.path.basename(test_file))
print("Filtered shape:", sample_train.shape)
print(
    "Day range:",
    sample_train["d"].min(),
    "to",
    sample_train["d"].max()
)
print(
    "Expected rows for this chunk:",
    80 * 3049
)

print(
    "\nMemory used:",
    round(sample_train.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)

print("\nMissing historical features:")
print(sample_train[history_cols].isna().sum())

Test file: CA_1_1400.parquet
Filtered shape: (243920, 25)
Day range: 1521 to 1600
Expected rows for this chunk: 243920

Memory used: 13.97 MB

Missing historical features:
sales_lag_1        0
sales_lag_7        0
sales_lag_28       0
rolling_mean_7     0
rolling_mean_28    0
dtype: int64


In [4]:
'''
align the categorical values
Before combining data from different stores, we need to make sure categories such as store_id, state_id, and event names use consistent category levels. Otherwise, Pandas may convert some categorical columns into ordinary strings when we concatenate the files, wasting memory.
We'll first collect the category levels from the 30 training source files, reading one file at a time.'''

import gc
import pandas as pd

categorical_features = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

# Collect category levels without retaining all file data
category_vocab = {
    col: set()
    for col in categorical_features
}

for i, file in enumerate(train_source_files, start=1):

    cat_df = pd.read_parquet(
        file,
        columns=categorical_features
    )

    for col in categorical_features:
        series = cat_df[col]

        if isinstance(series.dtype, pd.CategoricalDtype):
            values = series.cat.categories
        else:
            values = series.dropna().unique()

        category_vocab[col].update(
            str(value)
            for value in values
            if pd.notna(value)
        )

    del cat_df
    gc.collect()

    if i % 10 == 0:
        print(f"Processed {i}/{len(train_source_files)} files")

# Sort the shared category lists
category_vocab = {
    col: sorted(values)
    for col, values in category_vocab.items()
}

print("\nCategory levels collected:")
for col, values in category_vocab.items():
    print(f"{col}: {len(values)}")

Processed 10/30 files
Processed 20/30 files
Processed 30/30 files

Category levels collected:
item_id: 3049
dept_id: 7
cat_id: 3
store_id: 10
state_id: 3
event_name_1: 31
event_type_1: 5
event_name_2: 1
event_type_2: 1


In [5]:
#Verify the second event columns
event_counts = {
    "event_name_2": {},
    "event_type_2": {}
}

for file in train_source_files:
    temp = pd.read_parquet(
        file,
        columns=["d", "event_name_2", "event_type_2"]
    )

    # Keep only the training period
    temp = temp[
        temp["d"].between(TRAIN_START_D, TRAIN_END_D)
    ]

    for col in event_counts:
        counts = (
            temp[col]
            .astype("object")
            .value_counts(dropna=False)
        )

        for value, count in counts.items():
            value = str(value)
            event_counts[col][value] = (
                event_counts[col].get(value, 0) + int(count)
            )

    del temp
    gc.collect()

for col, counts in event_counts.items():
    print(f"\n{col}:")
    for value, count in sorted(counts.items()):
        print(f"  {value}: {count:,}")


event_name_2:
  No event: 11,128,850

event_type_2:
  No event: 11,128,850


In [6]:
import os
import glob

print("Current working directory:")
print(os.getcwd())

calendar_files = glob.glob(
    os.path.join("..", "data", "**", "calendar.csv"),
    recursive=True
)

print("\nCalendar files found:")
for path in calendar_files:
    print(path)

Current working directory:
D:\PIYUSH KUMAR MISHRA\JOB_PREPARATION\DataScience\MachineLearningProjects\ai-supply-chain-intelligence\notebooks

Calendar files found:
..\data\raw\calendar.csv


In [7]:
import pandas as pd

calendar_path = calendar_files[0]

calendar = pd.read_csv(calendar_path)

print("Calendar loaded successfully")
print("Shape:", calendar.shape)
print("Columns:", calendar.columns.tolist())

print("\nLast 5 rows:")
print(calendar.tail())

Calendar loaded successfully
Shape: (1969, 14)
Columns: ['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

Last 5 rows:
            date  wm_yr_wk    weekday  wday  month  year       d  \
1964  2016-06-15     11620  Wednesday     5      6  2016  d_1965   
1965  2016-06-16     11620   Thursday     6      6  2016  d_1966   
1966  2016-06-17     11620     Friday     7      6  2016  d_1967   
1967  2016-06-18     11621   Saturday     1      6  2016  d_1968   
1968  2016-06-19     11621     Sunday     2      6  2016  d_1969   

      event_name_1 event_type_1  event_name_2 event_type_2  snap_CA  snap_TX  \
1964           NaN          NaN           NaN          NaN        0        1   
1965           NaN          NaN           NaN          NaN        0        0   
1966           NaN          NaN           NaN          NaN        0        0   
1967           NaN          NaN           

In [8]:
# Inspect calendar information for the 28-day forecast horizon
'''
Before preparing the training data, let's check whether the future 28 days contain any second-event information. We should not drop those columns blindly if future dates contain events the model needs to handle.'''

calendar_check = calendar.copy()

calendar_check["d_num"] = (
    calendar_check["d"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(int)
)

future_calendar = calendar_check[
    calendar_check["d_num"].between(
        FORECAST_START_D,
        FORECAST_END_D
    )
].copy()

event_cols = [
    "d",
    "date",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

print("Future calendar rows:", len(future_calendar))

print("\nFuture events:")
print(
    future_calendar[event_cols].to_string(index=False)
)

print("\nSecond-event counts:")
print(
    future_calendar["event_name_2"]
    .fillna("No event")
    .value_counts(dropna=False)
)

Future calendar rows: 28

Future events:
     d       date   event_name_1 event_type_1 event_name_2 event_type_2
d_1914 2016-04-25            NaN          NaN          NaN          NaN
d_1915 2016-04-26            NaN          NaN          NaN          NaN
d_1916 2016-04-27            NaN          NaN          NaN          NaN
d_1917 2016-04-28            NaN          NaN          NaN          NaN
d_1918 2016-04-29            NaN          NaN          NaN          NaN
d_1919 2016-04-30     Pesach End    Religious          NaN          NaN
d_1920 2016-05-01 OrthodoxEaster    Religious          NaN          NaN
d_1921 2016-05-02            NaN          NaN          NaN          NaN
d_1922 2016-05-03            NaN          NaN          NaN          NaN
d_1923 2016-05-04            NaN          NaN          NaN          NaN
d_1924 2016-05-05  Cinco De Mayo     Cultural          NaN          NaN
d_1925 2016-05-06            NaN          NaN          NaN          NaN
d_1926 2016-05-07      

In [9]:
import psutil

memory = psutil.virtual_memory()

print(f"Total RAM: {memory.total / (1024**3):.2f} GB")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"Currently used: {memory.percent}%")

Total RAM: 5.83 GB
Available RAM: 0.63 GB
Currently used: 89.1%


In [10]:
import gc
import psutil

# Delete temporary inspection objects if they exist
for name in [
    "sample_train",
    "cat_df",
    "temp",
    "calendar_check",
    "future_calendar"
]:
    if name in globals():
        del globals()[name]

gc.collect()

memory = psutil.virtual_memory()

print(f"Total RAM: {memory.total / (1024**3):.2f} GB")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"RAM used: {memory.percent}%")

Total RAM: 5.83 GB
Available RAM: 0.64 GB
RAM used: 89.0%


In [11]:
import numpy as np

# Select dates throughout the training period
N_TRAIN_DAYS = 56

rng = np.random.default_rng(42)

all_training_days = np.arange(
    TRAIN_START_D,
    TRAIN_END_D + 1
)

# Divide the full period into 56 time intervals
date_bins = np.array_split(
    all_training_days,
    N_TRAIN_DAYS
)

# Select one date from each interval
train_sample_days = sorted(
    int(rng.choice(date_bin))
    for date_bin in date_bins
)

print("Number of selected dates:", len(train_sample_days))
print("First selected day:", train_sample_days[0])
print("Last selected day:", train_sample_days[-1])

print("\nSelected training dates:")
print(train_sample_days)

print(
    "\nExpected training rows:",
    len(train_sample_days) * 30_490
)

Number of selected dates: 56
First selected day: 1521
Last selected day: 1880

Selected training dates:
[1521, 1533, 1539, 1545, 1552, 1562, 1563, 1574, 1578, 1584, 1594, 1604, 1610, 1617, 1624, 1631, 1636, 1640, 1652, 1657, 1664, 1670, 1676, 1688, 1694, 1700, 1705, 1715, 1720, 1726, 1732, 1737, 1742, 1751, 1759, 1760, 1771, 1776, 1779, 1787, 1790, 1800, 1806, 1810, 1814, 1825, 1828, 1837, 1842, 1848, 1854, 1857, 1864, 1870, 1876, 1880]

Expected training rows: 1707440


In [12]:
import os
import glob
import gc
import pandas as pd

# Save the sampled training data separately
training_sample_folder = (
    "../data/processed/training_sample_56days"
)

os.makedirs(training_sample_folder, exist_ok=True)

# Remove previous outputs from this specific sampling step, if any
old_files = glob.glob(
    os.path.join(training_sample_folder, "train_*.parquet")
)

for path in old_files:
    os.remove(path)

# Read only the features, target and day identifier
required_columns = list(
    dict.fromkeys(
        feature_cols + [target_col, "d"]
    )
)

total_saved_rows = 0

for i, file in enumerate(train_source_files, start=1):

    # Read one source chunk at a time
    part = pd.read_parquet(
        file,
        columns=required_columns
    )

    # Keep only our 56 selected training dates
    part = part[
        part["d"].isin(train_sample_days)
    ].copy()

    # Align category levels across every saved partition
    for col in categorical_features:
        part[col] = pd.Categorical(
            part[col].astype("string"),
            categories=category_vocab[col]
        )

    rows = len(part)

    if rows > 0:
        output_name = "train_" + os.path.basename(file)
        output_path = os.path.join(
            training_sample_folder,
            output_name
        )

        part.to_parquet(
            output_path,
            index=False
        )

        total_saved_rows += rows

    del part
    gc.collect()

    if i % 10 == 0:
        print(
            f"Processed {i}/{len(train_source_files)} files | "
            f"Saved rows: {total_saved_rows:,}"
        )

expected_rows = len(train_sample_days) * 30_490

print("\nSampling completed.")
print(f"Total saved rows: {total_saved_rows:,}")
print(f"Expected rows:    {expected_rows:,}")
print(
    "Row count correct:",
    total_saved_rows == expected_rows
)
print(
    "Saved partitions:",
    len(glob.glob(
        os.path.join(training_sample_folder, "train_*.parquet")
    ))
)

Processed 10/30 files | Saved rows: 545,771
Processed 20/30 files | Saved rows: 1,152,522
Processed 30/30 files | Saved rows: 1,707,440

Sampling completed.
Total saved rows: 1,707,440
Expected rows:    1,707,440
Row count correct: True
Saved partitions: 30


In [13]:
'''
Now we need to check missing values before constructing the LightGBM dataset. We'll read one partition at a time to keep memory usage low.
Step 10 — Validate the sampled training data
'''

import os
import glob
import gc
import pandas as pd

training_sample_files = sorted(
    glob.glob(
        os.path.join(training_sample_folder, "train_*.parquet")
    )
)

required_columns = list(
    dict.fromkeys(feature_cols + [target_col, "d"])
)

total_rows = 0
daily_rows = {}
missing_counts = {col: 0 for col in required_columns}
largest_partition_mb = 0

for file in training_sample_files:
    part = pd.read_parquet(
        file,
        columns=required_columns
    )

    total_rows += len(part)

    # Count missing values without combining the partitions
    for col in required_columns:
        missing_counts[col] += int(part[col].isna().sum())

    # Verify the number of observations for each selected day
    for day, count in part.groupby("d", observed=True).size().items():
        daily_rows[int(day)] = (
            daily_rows.get(int(day), 0) + int(count)
        )

    partition_mb = (
        part.memory_usage(deep=True).sum() / (1024 ** 2)
    )
    largest_partition_mb = max(
        largest_partition_mb,
        partition_mb
    )

    del part
    gc.collect()

print("Partitions:", len(training_sample_files))
print(f"Total rows: {total_rows:,}")
print(f"Expected rows: {len(train_sample_days) * 30_490:,}")
print("Unique training days:", len(daily_rows))
print(f"Largest partition memory: {largest_partition_mb:.2f} MB")

print("\nRows per selected day:")
if daily_rows:
    print("Minimum:", min(daily_rows.values()))
    print("Maximum:", max(daily_rows.values()))

print(
    "All selected days contain 30,490 rows:",
    all(daily_rows.get(day, 0) == 30_490
        for day in train_sample_days)
)

print("\nMissing values:")
missing_series = pd.Series(missing_counts)
print(missing_series[missing_series > 0].sort_values(ascending=False))

Partitions: 30
Total rows: 1,707,440
Expected rows: 1,707,440
Unique training days: 56
Largest partition memory: 4.85 MB

Rows per selected day:
Minimum: 30490
Maximum: 30490
All selected days contain 30,490 rows: True

Missing values:
price_change     6511
price_changed    6511
sell_price       6470
dtype: int64


In [14]:
import gc
import pandas as pd
import numpy as np

train_frames = []

required_columns = feature_cols + [target_col]

for i, file in enumerate(training_sample_files, start=1):

    part = pd.read_parquet(
        file,
        columns=required_columns
    )

    # Ensure all partitions use the same categorical levels
    for col in categorical_features:
        part[col] = part[col].cat.set_categories(
            category_vocab[col]
        )

    # Normalize numeric price columns and their missing values
    for col in ["sell_price", "price_change", "price_changed"]:
        part[col] = pd.to_numeric(
            part[col],
            errors="coerce"
        ).astype("float32")

    # Keep the target compact for now
    part[target_col] = part[target_col].astype("float32")

    train_frames.append(part)

    if i % 10 == 0:
        print(f"Loaded {i}/{len(training_sample_files)} partitions")

    del part
    gc.collect()

# Combine the small partitions
train_df = pd.concat(
    train_frames,
    ignore_index=True,
    copy=False
)

del train_frames
gc.collect()

print("\nTraining data assembled.")
print("Shape:", train_df.shape)
print(f"Memory usage: {train_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("Missing target values:", train_df[target_col].isna().sum())
print("Missing price_changed values:", train_df["price_changed"].isna().sum())

Loaded 10/30 partitions
Loaded 20/30 partitions
Loaded 30/30 partitions

Training data assembled.
Shape: (1707440, 24)
Memory usage: 86.55 MB
Missing target values: 0
Missing price_changed values: 6511


In [15]:
'''
Prepare X_train and y_train
Remember:
- X_train contains the information the model learns from.
- y_train contains the actual sales values the model must predict.
We'll also remove constant features from this training sample, because a feature that never changes cannot help LightGBM split the data.
'''

import gc
import numpy as np
import pandas as pd

# 1. Separate features and target
X_train = train_df[feature_cols]
y_train = train_df[target_col]

# 2. Identify features with only one value
constant_cols = [
    col
    for col in X_train.columns
    if X_train[col].nunique(dropna=False) == 1
]

print("Constant features:", constant_cols)

# 3. Remove them from the training features
if constant_cols:
    X_train = X_train.drop(columns=constant_cols)

# 4. Update the categorical feature list
active_categorical_features = [
    col
    for col in categorical_features
    if col in X_train.columns
]

# 5. Verify categorical dtypes
print("\nCategorical columns:")
for col in active_categorical_features:
    print(f"{col}: {X_train[col].dtype}")

# 6. Check dimensions and missing values
print("\nX_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("Missing target values:", y_train.isna().sum())

print(
    "Missing numerical feature values:",
    int(X_train.select_dtypes(include="number").isna().sum().sum())
)

print(
    f"X_train memory: "
    f"{X_train.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Constant features: ['event_name_2', 'event_type_2']

Categorical columns:
item_id: category
dept_id: category
cat_id: category
store_id: category
state_id: category
event_name_1: category
event_type_1: category

X_train shape: (1707440, 21)
y_train shape: (1707440,)
Missing target values: 0
Missing numerical feature values: 19492
X_train memory: 76.78 MB


In [16]:
'''
Prepare the 28-day validation history
To simulate a genuine 28-day forecast, we need two periods:
- History: d_1858 to d_1885 — the previous 28 days of actual sales.
- Validation: d_1886 to d_1913 — the 28 days we will predict
'''
import gc
import pandas as pd
import numpy as np

# Release the original combined training table.
# X_train and y_train have already been created.
if "train_df" in globals():
    del train_df

gc.collect()

# Last 28 days before the validation period
HISTORY_START_D = VALID_START_D - 28

# The active model features, excluding lag/rolling columns.
active_feature_cols = X_train.columns.tolist()

static_feature_cols = [
    col for col in active_feature_cols
    if col not in history_cols
]

# We need series identity, day, target, and non-lag features.
validation_columns = list(dict.fromkeys(
    ["id", "d", "sales"] + static_feature_cols
))

validation_frames = []

for i, file in enumerate(valid_source_files, start=1):

    part = pd.read_parquet(
        file,
        columns=validation_columns
    )

    # Keep 28 days of history and 28 validation days
    part = part[
        part["d"].between(HISTORY_START_D, VALID_END_D)
    ].copy()

    # Align categories with the training data
    for col in active_categorical_features:
        if col in part.columns:
            part[col] = pd.Categorical(
                part[col].astype("string"),
                categories=category_vocab[col]
            )

    # Keep missing prices as NaN
    for col in ["sell_price", "price_change", "price_changed"]:
        if col in part.columns:
            part[col] = pd.to_numeric(
                part[col],
                errors="coerce"
            ).astype("float32")

    part["sales"] = part["sales"].astype("float32")

    validation_frames.append(part)

    if i % 5 == 0:
        print(
            f"Loaded {i}/{len(valid_source_files)} "
            "validation source files"
        )

# Combine only this 56-day window
validation_work = pd.concat(
    validation_frames,
    ignore_index=True,
    copy=False
)

del validation_frames
gc.collect()

# Compact the series identifier
validation_work["id"] = validation_work["id"].astype("category")

# Sort for subsequent history-based calculations
validation_work = validation_work.sort_values(
    ["id", "d"]
).reset_index(drop=True)

print("\nValidation preparation data:")
print("Shape:", validation_work.shape)
print(
    f"Day range: d_{validation_work['d'].min()} "
    f"to d_{validation_work['d'].max()}"
)
print("Unique item-store series:", validation_work["id"].nunique())

print(
    f"Memory usage: "
    f"{validation_work.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

print(
    "Expected rows:",
    30_490 * 56
)
print(
    "Missing sales:",
    validation_work["sales"].isna().sum()
)

Loaded 5/10 validation source files
Loaded 10/10 validation source files

Validation preparation data:
Shape: (1707440, 19)
Day range: d_1858 to d_1913
Unique item-store series: 30490
Memory usage: 60.51 MB
Expected rows: 1707440
Missing sales: 0


In [17]:
import numpy as np
import pandas as pd
import gc

# Recover the day associated with each row in X_train.
# Read only the small "d" column from each saved partition.
train_day_parts = []

for file in training_sample_files:
    part_days = pd.read_parquet(file, columns=["d"])
    train_day_parts.append(part_days["d"].to_numpy())

train_row_days = np.concatenate(train_day_parts)

del train_day_parts
gc.collect()

# Confirm row ordering matches X_train and y_train
assert len(train_row_days) == len(X_train) == len(y_train)

# Reserve the latest 20% of sampled dates for early stopping
split_idx = int(len(train_sample_days) * 0.8)

fit_days = train_sample_days[:split_idx]
early_stopping_days = train_sample_days[split_idx:]

fit_mask = np.isin(train_row_days, fit_days)
early_stop_mask = np.isin(
    train_row_days,
    early_stopping_days
)

print("Training dates:", len(fit_days))
print("Early-stopping dates:", len(early_stopping_days))

print(
    f"Training rows: {fit_mask.sum():,}"
)
print(
    f"Early-stopping rows: {early_stop_mask.sum():,}"
)

print(
    "\nTraining day range:",
    min(fit_days), "to", max(fit_days)
)
print(
    "Early-stopping day range:",
    min(early_stopping_days), "to", max(early_stopping_days)
)

print(
    "\nMasks cover all rows:",
    np.all(fit_mask | early_stop_mask)
)
print(
    "Masks overlap:",
    np.any(fit_mask & early_stop_mask)
)

Training dates: 44
Early-stopping dates: 12
Training rows: 1,341,560
Early-stopping rows: 365,880

Training day range: 1521 to 1810
Early-stopping day range: 1814 to 1880

Masks cover all rows: True
Masks overlap: False


In [18]:
import gc
import psutil
import lightgbm as lgb

gc.collect()

memory = psutil.virtual_memory()

print("LightGBM version:", lgb.__version__)
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"RAM currently used: {memory.percent}%")

print("\nData sizes:")
print("Training feature matrix:", X_train.shape)
print("Training target:", y_train.shape)
print("Validation preparation data:", validation_work.shape)
print("Categorical features:", active_categorical_features)

LightGBM version: 4.7.0
Available RAM: 0.52 GB
RAM currently used: 91.1%

Data sizes:
Training feature matrix: (1707440, 21)
Training target: (1707440,)
Validation preparation data: (1707440, 19)
Categorical features: ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'event_name_1', 'event_type_1']


In [19]:
import gc
import psutil

# Create chronological subsets
X_fit = X_train.loc[fit_mask].copy()
y_fit = y_train.loc[fit_mask].copy()

X_es = X_train.loc[early_stop_mask].copy()
y_es = y_train.loc[early_stop_mask].copy()

# Release the original combined matrices
del X_train, y_train
del train_row_days, fit_mask, early_stop_mask

gc.collect()

# Verify the resulting datasets
print("Fitting data:", X_fit.shape, y_fit.shape)
print("Early-stopping data:", X_es.shape, y_es.shape)

print(
    "\nCategorical dtypes preserved:",
    all(
        str(X_fit[col].dtype) == "category"
        and str(X_es[col].dtype) == "category"
        for col in active_categorical_features
    )
)

memory = psutil.virtual_memory()
print(f"\nAvailable RAM: {memory.available / (1024**3):.2f} GB")
print(f"RAM usage: {memory.percent}%")

Fitting data: (1341560, 21) (1341560,)
Early-stopping data: (365880, 21) (365880,)

Categorical dtypes preserved: True

Available RAM: 0.53 GB
RAM usage: 91.0%


In [20]:
import lightgbm as lgb
import gc
import psutil

# Use reduced binning for a memory-conscious baseline
dataset_params = {
    "max_bin": 63
}

print("Building LightGBM training dataset...")

train_data = lgb.Dataset(
    X_fit,
    label=y_fit,
    categorical_feature=active_categorical_features,
    params=dataset_params,
    free_raw_data=True
)

train_data.construct()

memory = psutil.virtual_memory()
print("Training dataset constructed.")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")


print("\nBuilding early-stopping dataset...")

valid_data = lgb.Dataset(
    X_es,
    label=y_es,
    categorical_feature=active_categorical_features,
    reference=train_data,
    params=dataset_params,
    free_raw_data=True
)

valid_data.construct()

gc.collect()

memory = psutil.virtual_memory()
print("Early-stopping dataset constructed.")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")

print("\nTraining rows:", train_data.num_data())
print("Early-stopping rows:", valid_data.num_data())
print("Features:", train_data.num_feature())

Building LightGBM training dataset...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
Training dataset constructed.
Available RAM: 0.50 GB

Building early-stopping dataset...
Early-stopping dataset constructed.
Available RAM: 0.52 GB

Training rows: 1341560
Early-stopping rows: 365880
Features: 21


In [21]:
import gc
import psutil

# LightGBM datasets have already been constructed.
# The raw pandas matrices are no longer needed for fitting.
for name in ["X_fit", "y_fit", "X_es", "y_es"]:
    if name in globals():
        del globals()[name]

gc.collect()

memory = psutil.virtual_memory()

print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"RAM usage: {memory.percent}%")

print("\nLightGBM datasets remain available:")
print("Training rows:", train_data.num_data())
print("Early-stopping rows:", valid_data.num_data())
print("Features:", train_data.num_feature())

Available RAM: 0.64 GB
RAM usage: 89.0%

LightGBM datasets remain available:
Training rows: 1341560
Early-stopping rows: 365880
Features: 21


In [22]:
import lightgbm as lgb
import gc
import psutil

params = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.1,
    "metric": "rmse",

    # Conservative model complexity
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 100,
    "max_bin": 63,

    # Memory and compute settings
    "num_threads": 2,
    "force_col_wise": True,

    "seed": 42,
    "verbosity": -1
}

print("Starting LightGBM Tweedie training...")
print(f"Available RAM before training: "
      f"{psutil.virtual_memory().available / (1024**3):.2f} GB")

model = lgb.train(
    params=params,
    train_set=train_data,
    num_boost_round=800,
    valid_sets=[valid_data],
    valid_names=["early_stop"],
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=40,
            first_metric_only=True
        ),
        lgb.log_evaluation(period=25)
    ]
)

gc.collect()

print("\nTraining completed.")
print("Best iteration:", model.best_iteration)
print("Best validation RMSE:", model.best_score)
print(f"Available RAM after training: "
      f"{psutil.virtual_memory().available / (1024**3):.2f} GB")

Starting LightGBM Tweedie training...
Available RAM before training: 0.63 GB
Training until validation scores don't improve for 40 rounds
[25]	early_stop's rmse: 2.4125
[50]	early_stop's rmse: 2.11275
[75]	early_stop's rmse: 2.06501
[100]	early_stop's rmse: 2.05375
[125]	early_stop's rmse: 2.04969
[150]	early_stop's rmse: 2.04894
[175]	early_stop's rmse: 2.04883
[200]	early_stop's rmse: 2.04865
Early stopping, best iteration is:
[170]	early_stop's rmse: 2.04842
Evaluated only: rmse

Training completed.
Best iteration: 170
Best validation RMSE: defaultdict(<class 'collections.OrderedDict'>, {'early_stop': OrderedDict({'rmse': 2.048423509547645})})
Available RAM after training: 0.85 GB


In [23]:
import numpy as np
import gc

# Each series needs exactly 56 daily observations
n_series = validation_work["id"].cat.categories.size
n_days = VALID_END_D - HISTORY_START_D + 1

# Map each row to its item-store series and relative day
series_codes = validation_work["id"].cat.codes.to_numpy(
    dtype=np.int32
)

day_offsets = (
    validation_work["d"].to_numpy(dtype=np.int32)
    - HISTORY_START_D
)

sales_values = validation_work["sales"].to_numpy(
    dtype=np.float32
)

# Build a compact matrix: rows = series, columns = days
sales_matrix = np.full(
    (n_series, n_days),
    np.nan,
    dtype=np.float32
)

sales_matrix[series_codes, day_offsets] = sales_values

# Separate history from the actual validation targets
history_length = VALID_START_D - HISTORY_START_D

history_sales = sales_matrix[:, :history_length].copy()
actual_validation_sales = sales_matrix[:, history_length:].copy()

print("Number of series:", n_series)
print("Sales matrix shape:", sales_matrix.shape)

print(
    "Historical sales shape:",
    history_sales.shape
)
print(
    "Actual validation sales shape:",
    actual_validation_sales.shape
)

print(
    "Missing cells in sales matrix:",
    np.isnan(sales_matrix).sum()
)

print(
    "History dates:",
    f"d_{HISTORY_START_D} to d_{VALID_START_D - 1}"
)
print(
    "Validation dates:",
    f"d_{VALID_START_D} to d_{VALID_END_D}"
)

assert n_series == 30_490
assert history_sales.shape == (30_490, 28)
assert actual_validation_sales.shape == (30_490, 28)
assert not np.isnan(sales_matrix).any()

print("\nValidation history prepared successfully.")

Number of series: 30490
Sales matrix shape: (30490, 56)
Historical sales shape: (30490, 28)
Actual validation sales shape: (30490, 28)
Missing cells in sales matrix: 0
History dates: d_1858 to d_1885
Validation dates: d_1886 to d_1913

Validation history prepared successfully.


In [24]:
'''
Test the first forecast day
Before predicting all 28 days, we'll test just day d_1886.
For every product-store combination, we'll construct the five historical features using only sales through d_1885, then ask the trained LightGBM model to predict sales for d_1886.
'''
import numpy as np
import pandas as pd

# 1. Select the first validation day
day_df = validation_work[
    validation_work["d"] == VALID_START_D
].copy()

# 2. Ensure the rows follow the exact same series order
series_order = day_df["id"].cat.codes.to_numpy()

assert len(day_df) == 30_490
assert np.array_equal(
    series_order,
    np.arange(30_490)
), "Series order does not match the history matrix!"

# 3. Build the static features for this day
X_day = day_df[static_feature_cols].copy()

# 4. Construct demand history features from d_1858 to d_1885
X_day["sales_lag_1"] = history_sales[:, -1]
X_day["sales_lag_7"] = history_sales[:, -7]
X_day["sales_lag_28"] = history_sales[:, -28]

X_day["rolling_mean_7"] = history_sales[:, -7:].mean(axis=1)
X_day["rolling_mean_28"] = history_sales[:, -28:].mean(axis=1)

# 5. Match the exact feature order used during training
X_day = X_day[active_feature_cols]

assert X_day.columns.tolist() == model.feature_name()

# 6. Predict day d_1886
first_day_predictions = model.predict(
    X_day,
    num_iteration=model.best_iteration
)

first_day_predictions = np.asarray(
    first_day_predictions,
    dtype=np.float32
)

print("Forecast day:", VALID_START_D)
print("Feature matrix shape:", X_day.shape)
print("Number of predictions:", len(first_day_predictions))

print(
    "\nPrediction summary:"
)
print("Minimum:", first_day_predictions.min())
print("Maximum:", first_day_predictions.max())
print("Mean:", first_day_predictions.mean())

print(
    "\nActual sales mean for this day:",
    actual_validation_sales[:, 0].mean()
)

print("\nFirst 10 predictions:")
print(first_day_predictions[:10])

Forecast day: 1886
Feature matrix shape: (30490, 21)
Number of predictions: 30490

Prediction summary:
Minimum: 0.005721811
Maximum: 78.577065
Mean: 1.2272073

Actual sales mean for this day: 1.1820596

First 10 predictions:
[0.63172674 0.68057656 0.89374566 0.5109478  0.64470917 0.50752765
 0.52873623 0.38641834 0.35860693 0.325579  ]


In [25]:
#Forecast all 28 validation days
import numpy as np
import gc
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Store predictions for all 30,490 series across 28 days
recursive_predictions = np.zeros(
    (30_490, HORIZON),
    dtype=np.float32
)

# Start from actual sales through d_1885
recursive_history = history_sales.copy()

for step, day in enumerate(
    range(VALID_START_D, VALID_END_D + 1)
):

    # Get calendar, product, store, price and event features for this day
    day_df = validation_work[
        validation_work["d"] == day
    ].copy()

    # Verify that series order matches our history matrix
    series_codes = day_df["id"].cat.codes.to_numpy(
        dtype=np.int32
    )

    assert len(day_df) == 30_490
    assert np.array_equal(
        series_codes,
        np.arange(30_490)
    ), f"Series order mismatch on day {day}"

    # Keep non-history features
    X_day = day_df[static_feature_cols].copy()

    # Recalculate demand history features
    X_day["sales_lag_1"] = recursive_history[:, -1]
    X_day["sales_lag_7"] = recursive_history[:, -7]
    X_day["sales_lag_28"] = recursive_history[:, -28]

    X_day["rolling_mean_7"] = recursive_history[:, -7:].mean(axis=1)
    X_day["rolling_mean_28"] = recursive_history.mean(axis=1)

    # Match training feature order
    X_day = X_day[active_feature_cols]

    # Predict this day
    day_predictions = model.predict(
        X_day,
        num_iteration=model.best_iteration
    ).astype(np.float32)

    recursive_predictions[:, step] = day_predictions

    # Feed predictions into history for the NEXT day.
    # Actual validation sales are never used here.
    recursive_history = np.concatenate(
        [
            recursive_history[:, 1:],
            day_predictions.reshape(-1, 1)
        ],
        axis=1
    )

    if (step + 1) % 7 == 0 or step == 0:
        print(
            f"Completed {step + 1}/{HORIZON} days | "
            f"d_{day} | "
            f"mean prediction={day_predictions.mean():.4f}"
        )

    del day_df, X_day, day_predictions
    gc.collect()

# Score only after all predictions are generated
actual = actual_validation_sales.ravel()
predicted = recursive_predictions.ravel()

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))
wape = (
    np.abs(actual - predicted).sum()
    / actual.sum()
    * 100
)

print("\n--- 28-Day Recursive Validation Results ---")
print(f"Forecasted observations: {len(predicted):,}")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"WAPE: {wape:.2f}%")
print(f"Mean actual sales: {actual.mean():.4f}")
print(f"Mean predicted sales: {predicted.mean():.4f}")

Completed 1/28 days | d_1886 | mean prediction=1.2272
Completed 7/28 days | d_1892 | mean prediction=1.7257
Completed 14/28 days | d_1899 | mean prediction=1.7219
Completed 21/28 days | d_1906 | mean prediction=1.7240
Completed 28/28 days | d_1913 | mean prediction=1.7522

--- 28-Day Recursive Validation Results ---
Forecasted observations: 853,720
MAE:  1.0661
RMSE: 2.2090
WAPE: 76.89%
Mean actual sales: 1.3864
Mean predicted sales: 1.4283


In [26]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --------------------------------------------------
# 1. Seasonal-naive recursive forecast
# Predict each day using the same weekday from 7 days earlier
# --------------------------------------------------

naive_predictions = np.zeros(
    (30_490, HORIZON),
    dtype=np.float32
)

naive_history = history_sales.copy()

for step in range(HORIZON):
    day_prediction = naive_history[:, -7].copy()

    naive_predictions[:, step] = day_prediction

    naive_history = np.concatenate(
        [
            naive_history[:, 1:],
            day_prediction.reshape(-1, 1)
        ],
        axis=1
    )

# --------------------------------------------------
# 2. Overall metric comparison
# --------------------------------------------------

def calculate_metrics(actual, predicted):
    actual = np.asarray(actual).ravel()
    predicted = np.asarray(predicted).ravel()

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    wape = (
        np.abs(actual - predicted).sum()
        / np.abs(actual).sum()
        * 100
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "WAPE (%)": wape
    }

model_metrics = calculate_metrics(
    actual_validation_sales,
    recursive_predictions
)

naive_metrics = calculate_metrics(
    actual_validation_sales,
    naive_predictions
)

comparison = pd.DataFrame({
    "LightGBM": model_metrics,
    "Seasonal naive": naive_metrics
})

print("Overall 28-day recursive comparison:")
print(comparison.round(4))

# --------------------------------------------------
# 3. Compare the error by forecast day
# --------------------------------------------------

daily_results = []

for step in range(HORIZON):
    actual_day = actual_validation_sales[:, step]
    model_day = recursive_predictions[:, step]
    naive_day = naive_predictions[:, step]

    daily_results.append({
        "day": VALID_START_D + step,
        "actual_mean": actual_day.mean(),
        "lgb_mean": model_day.mean(),
        "naive_mean": naive_day.mean(),
        "lgb_mae": mean_absolute_error(actual_day, model_day),
        "naive_mae": mean_absolute_error(actual_day, naive_day)
    })

daily_comparison = pd.DataFrame(daily_results)

print("\nFirst 7 forecast days:")
print(daily_comparison.head(7).round(4).to_string(index=False))

print("\nLast 7 forecast days:")
print(daily_comparison.tail(7).round(4).to_string(index=False))

Overall 28-day recursive comparison:
          LightGBM  Seasonal naive
MAE         1.0661          1.2324
RMSE        2.2090          2.7595
WAPE (%)   76.8946         88.8867

First 7 forecast days:
 day  actual_mean  lgb_mean  naive_mean  lgb_mae  naive_mae
1886       1.1821    1.2272      1.2253   0.9053     1.1056
1887       1.1104    1.2401      1.1635   0.9156     1.0676
1888       1.0613    1.2400      1.1409   0.8985     1.0491
1889       1.1375    1.2431      1.1152   0.9299     1.0529
1890       1.4935    1.4958      1.4959   1.0793     1.2669
1891       1.7275    1.7423      1.7666   1.1778     1.4364
1892       1.8506    1.7257      1.5205   1.2424     1.4385

Last 7 forecast days:
 day  actual_mean  lgb_mean  naive_mean  lgb_mae  naive_mae
1907       1.2482    1.3032      1.2253   1.0282     1.1642
1908       1.2322    1.2549      1.1635   0.9997     1.1373
1909       1.1592    1.2488      1.1409   0.9920     1.1180
1910       1.1490    1.2485      1.1152   0.9839     1.0

In [27]:
import numpy as np
import pandas as pd

horizon_results = []

for step in range(HORIZON):
    actual_day = actual_validation_sales[:, step]
    lgb_day = recursive_predictions[:, step]
    naive_day = naive_predictions[:, step]

    horizon_results.append({
        "forecast_day": step + 1,
        "d": VALID_START_D + step,
        "actual_mean": actual_day.mean(),
        "lgb_mean": lgb_day.mean(),
        "lgb_bias": (lgb_day - actual_day).mean(),
        "lgb_mae": np.abs(lgb_day - actual_day).mean(),
        "lgb_rmse": np.sqrt(np.mean((lgb_day - actual_day) ** 2)),
        "lgb_wape_pct": (
            np.abs(lgb_day - actual_day).sum()
            / np.abs(actual_day).sum()
            * 100
        ),
        "naive_mae": np.abs(naive_day - actual_day).mean()
    })

horizon_results = pd.DataFrame(horizon_results)

print("First 5 forecast days:")
print(horizon_results.head().round(4).to_string(index=False))

print("\nLast 5 forecast days:")
print(horizon_results.tail().round(4).to_string(index=False))

print("\nAverage MAE, days 1–7:",
      round(horizon_results.iloc[:7]["lgb_mae"].mean(), 4))

print("Average MAE, days 22–28:",
      round(horizon_results.iloc[21:]["lgb_mae"].mean(), 4))

print("\nAverage bias, days 1–7:",
      round(horizon_results.iloc[:7]["lgb_bias"].mean(), 4))

print("Average bias, days 22–28:",
      round(horizon_results.iloc[21:]["lgb_bias"].mean(), 4))

First 5 forecast days:
 forecast_day    d  actual_mean  lgb_mean  lgb_bias  lgb_mae  lgb_rmse  lgb_wape_pct  naive_mae
            1 1886       1.1821    1.2272    0.0451   0.9053    1.8307       76.5835     1.1056
            2 1887       1.1104    1.2401    0.1297   0.9156    1.8942       82.4583     1.0676
            3 1888       1.0613    1.2400    0.1787   0.8985    1.7240       84.6594     1.0491
            4 1889       1.1375    1.2431    0.1056   0.9299    1.8029       81.7537     1.0529
            5 1890       1.4935    1.4958    0.0023   1.0793    2.0740       72.2668     1.2669

Last 5 forecast days:
 forecast_day    d  actual_mean  lgb_mean  lgb_bias  lgb_mae  lgb_rmse  lgb_wape_pct  naive_mae
           24 1909       1.1592    1.2488    0.0896   0.9920    1.9686       85.5795     1.1180
           25 1910       1.1490    1.2485    0.0995   0.9839    1.9432       85.6351     1.0813
           26 1911       1.3289    1.5258    0.1969   1.0800    2.1094       81.2709     1

In [28]:
import pandas as pd
import numpy as np

importance_df = pd.DataFrame({
    "feature": model.feature_name(),
    "gain": model.feature_importance(
        importance_type="gain",
        iteration=model.best_iteration
    ),
    "splits": model.feature_importance(
        importance_type="split",
        iteration=model.best_iteration
    )
})

total_gain = importance_df["gain"].sum()

importance_df["gain_share_pct"] = (
    importance_df["gain"] / total_gain * 100
    if total_gain > 0
    else 0.0
)

importance_df = importance_df.sort_values(
    "gain",
    ascending=False
).reset_index(drop=True)

print("Feature importance — best iteration:", model.best_iteration)

print(
    importance_df.round({
        "gain": 2,
        "gain_share_pct": 2
    }).to_string(index=False)
)

Feature importance — best iteration: 170
        feature        gain  splits  gain_share_pct
 rolling_mean_7 21916617.09     301           68.13
rolling_mean_28  8180486.81     433           25.43
        item_id  1143084.46    3297            3.55
    sales_lag_1   442440.02     204            1.38
    day_of_week   232372.67     302            0.72
   sales_lag_28    67209.87      55            0.21
    sales_lag_7    38929.55      35            0.12
        dept_id    36506.25      52            0.11
       store_id    26965.13     114            0.08
          month    20585.59      69            0.06
     sell_price    20271.55      76            0.06
   event_name_1    15942.85      65            0.05
         cat_id     7808.12      11            0.02
       state_id     6798.12      30            0.02
       is_event     5195.48      14            0.02
           year     3050.71      11            0.01
   price_change     1947.16      21            0.01
     is_weekend     142

In [29]:
import numpy as np
import pandas as pd

actual = actual_validation_sales.ravel()
predicted = recursive_predictions.ravel()

# Separate observations by actual demand
zero_mask = actual == 0
positive_mask = actual > 0

absolute_errors = np.abs(actual - predicted)
total_absolute_error = absolute_errors.sum()

zero_error = absolute_errors[zero_mask].sum()
positive_error = absolute_errors[positive_mask].sum()

diagnostics = pd.DataFrame({
    "Metric": [
        "Number of observations",
        "Percentage of observations",
        "Mean actual sales",
        "Mean predicted sales",
        "MAE",
        "Share of total absolute error (%)"
    ],
    "Zero sales": [
        zero_mask.sum(),
        zero_mask.mean() * 100,
        actual[zero_mask].mean(),
        predicted[zero_mask].mean(),
        absolute_errors[zero_mask].mean(),
        zero_error / total_absolute_error * 100
    ],
    "Positive sales": [
        positive_mask.sum(),
        positive_mask.mean() * 100,
        actual[positive_mask].mean(),
        predicted[positive_mask].mean(),
        absolute_errors[positive_mask].mean(),
        positive_error / total_absolute_error * 100
    ]
})

print(diagnostics.round(4).to_string(index=False))

                           Metric  Zero sales  Positive sales
           Number of observations 480460.0000     373260.0000
       Percentage of observations     56.2784         43.7216
                Mean actual sales      0.0000          3.1710
             Mean predicted sales      0.6700          2.4044
                              MAE      0.6700          1.5759
Share of total absolute error (%)     35.3688         64.6312


In [30]:
import numpy as np
import pandas as pd

actual = actual_validation_sales.ravel()
predicted = recursive_predictions.ravel()

# Classify observations by actual daily sales
demand_groups = {
    "Zero sales": actual == 0,
    "1 unit": actual == 1,
    "2–3 units": (actual >= 2) & (actual <= 3),
    "4–7 units": (actual >= 4) & (actual <= 7),
    "8+ units": actual >= 8
}

total_abs_error = np.abs(actual - predicted).sum()

results = []

for group_name, mask in demand_groups.items():
    group_actual = actual[mask]
    group_predicted = predicted[mask]
    group_abs_error = np.abs(group_actual - group_predicted)

    results.append({
        "Demand group": group_name,
        "Observations": int(mask.sum()),
        "Actual mean": group_actual.mean() if mask.any() else np.nan,
        "Predicted mean": group_predicted.mean() if mask.any() else np.nan,
        "Bias": (group_predicted - group_actual).mean() if mask.any() else np.nan,
        "MAE": group_abs_error.mean() if mask.any() else np.nan,
        "Share of total error (%)": (
            group_abs_error.sum() / total_abs_error * 100
            if total_abs_error > 0 else np.nan
        )
    })

demand_diagnostics = pd.DataFrame(results)

print(demand_diagnostics.round(4).to_string(index=False))

Demand group  Observations  Actual mean  Predicted mean    Bias    MAE  Share of total error (%)
  Zero sales        480460       0.0000          0.6700  0.6700 0.6700                   35.3688
      1 unit        163208       1.0000          1.0174  0.0174 0.6357                   11.4001
   2–3 units        122229       2.3433          1.7352 -0.6080 1.2851                   17.2585
   4–7 units         58445       4.9951          3.4887 -1.5064 2.4926                   16.0065
    8+ units         29378      15.0474         10.7364 -4.3111 6.1856                   19.9660


In [32]:
import gc
import os
import glob
import numpy as np
import psutil

# Release LightGBM's old binned datasets.
# The trained baseline model and its predictions remain available.
for name in ["train_data", "valid_data"]:
    if name in globals():
        del globals()[name]

gc.collect()

# Sample 112 dates across the same 365-day training window
N_TRAIN_DAYS_112 = 112

rng = np.random.default_rng(42)

all_training_days = np.arange(
    TRAIN_START_D,
    TRAIN_END_D + 1
)

date_bins_112 = np.array_split(
    all_training_days,
    N_TRAIN_DAYS_112
)

train_sample_days_112 = sorted(
    int(rng.choice(date_bin))
    for date_bin in date_bins_112
)

# Keep this experiment separate from the original 56-day sample
training_sample_112_folder = (
    "../data/processed/training_sample_112days"
)

os.makedirs(
    training_sample_112_folder,
    exist_ok=True
)

print("Selected training dates:", len(train_sample_days_112))
print("First selected day:", train_sample_days_112[0])
print("Last selected day:", train_sample_days_112[-1])

print(
    "Expected training rows:",
    len(train_sample_days_112) * 30_490
)

print(
    "Available RAM:",
    round(psutil.virtual_memory().available / (1024**3), 2),
    "GB"
)

Selected training dates: 112
First selected day: 1521
Last selected day: 1885
Expected training rows: 3414880
Available RAM: 0.79 GB


In [33]:
import os
import glob
import gc
import pandas as pd
import psutil

os.makedirs(training_sample_112_folder, exist_ok=True)

# Remove only the previous outputs of this experiment
old_files = glob.glob(
    os.path.join(training_sample_112_folder, "train_*.parquet")
)

for path in old_files:
    os.remove(path)

required_columns_112 = list(
    dict.fromkeys(feature_cols + [target_col, "d"])
)

total_saved_rows_112 = 0

for i, file in enumerate(train_source_files, start=1):

    # Read one source partition
    part = pd.read_parquet(
        file,
        columns=required_columns_112
    )

    # Keep the 112 selected dates
    part = part[
        part["d"].isin(train_sample_days_112)
    ].copy()

    # Align categories across all saved partitions
    for col in categorical_features:
        part[col] = pd.Categorical(
            part[col].astype("string"),
            categories=category_vocab[col]
        )

    rows = len(part)

    if rows:
        output_path = os.path.join(
            training_sample_112_folder,
            "train_" + os.path.basename(file)
        )

        part.to_parquet(output_path, index=False)
        total_saved_rows_112 += rows

    del part
    gc.collect()

    if i % 5 == 0 or i == len(train_source_files):
        available_gb = (
            psutil.virtual_memory().available / (1024**3)
        )
        print(
            f"Processed {i}/{len(train_source_files)} files | "
            f"Rows saved: {total_saved_rows_112:,} | "
            f"Available RAM: {available_gb:.2f} GB"
        )

expected_rows_112 = len(train_sample_days_112) * 30_490

print("\nSampling complete.")
print(f"Rows saved:    {total_saved_rows_112:,}")
print(f"Rows expected: {expected_rows_112:,}")
print(
    "Row count correct:",
    total_saved_rows_112 == expected_rows_112
)
print(
    "Saved partitions:",
    len(glob.glob(
        os.path.join(training_sample_112_folder, "train_*.parquet")
    ))
)

Processed 5/30 files | Rows saved: 594,555 | Available RAM: 0.86 GB
Processed 10/30 files | Rows saved: 1,085,444 | Available RAM: 0.85 GB
Processed 15/30 files | Rows saved: 1,707,440 | Available RAM: 0.86 GB
Processed 20/30 files | Rows saved: 2,301,995 | Available RAM: 0.86 GB
Processed 25/30 files | Rows saved: 2,792,884 | Available RAM: 0.84 GB
Processed 30/30 files | Rows saved: 3,414,880 | Available RAM: 0.85 GB

Sampling complete.
Rows saved:    3,414,880
Rows expected: 3,414,880
Row count correct: True
Saved partitions: 30


In [34]:
import gc
import psutil
import glob
import os

gc.collect()

memory = psutil.virtual_memory()

expanded_files = glob.glob(
    os.path.join(
        training_sample_112_folder,
        "train_*.parquet"
    )
)

print("Expanded sample partitions:", len(expanded_files))
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"RAM usage: {memory.percent}%")

Expanded sample partitions: 30
Available RAM: 0.94 GB
RAM usage: 83.9%


In [35]:
import numpy as np
import psutil

# Dates are already sorted chronologically
split_idx_112 = int(len(train_sample_days_112) * 0.8)

fit_days_112 = train_sample_days_112[:split_idx_112]
early_stop_days_112 = train_sample_days_112[split_idx_112:]

print("Fitting dates:", len(fit_days_112))
print("Early-stopping dates:", len(early_stop_days_112))

print(
    f"\nFitting day range: "
    f"d_{min(fit_days_112)} to d_{max(fit_days_112)}"
)

print(
    f"Early-stopping day range: "
    f"d_{min(early_stop_days_112)} to d_{max(early_stop_days_112)}"
)

print(
    "\nExpected fitting rows:",
    len(fit_days_112) * 30_490
)

print(
    "Expected early-stopping rows:",
    len(early_stop_days_112) * 30_490
)

print(
    "Chronological split valid:",
    max(fit_days_112) < min(early_stop_days_112)
)

print(
    f"\nAvailable RAM: "
    f"{psutil.virtual_memory().available / (1024**3):.2f} GB"
)

Fitting dates: 89
Early-stopping dates: 23

Fitting day range: d_1521 to d_1816
Early-stopping day range: d_1819 to d_1885

Expected fitting rows: 2713610
Expected early-stopping rows: 701270
Chronological split valid: True

Available RAM: 1.01 GB


In [36]:
import os
import glob
import gc
import pandas as pd

fit_folder_112 = (
    "../data/processed/training_112_fit"
)
early_stop_folder_112 = (
    "../data/processed/training_112_early_stop"
)

os.makedirs(fit_folder_112, exist_ok=True)
os.makedirs(early_stop_folder_112, exist_ok=True)

# Remove outputs from a previous run of this split step only
for folder in [fit_folder_112, early_stop_folder_112]:
    for path in glob.glob(os.path.join(folder, "*.parquet")):
        os.remove(path)

fit_rows_saved = 0
early_stop_rows_saved = 0

for i, file in enumerate(expanded_files, start=1):

    part = pd.read_parquet(file)

    # Separate the two chronological groups
    fit_part = part[
        part["d"].isin(fit_days_112)
    ].copy()

    early_part = part[
        part["d"].isin(early_stop_days_112)
    ].copy()

    filename = os.path.basename(file).replace(
        "train_", ""
    )

    if not fit_part.empty:
        fit_part.to_parquet(
            os.path.join(fit_folder_112, "fit_" + filename),
            index=False
        )
        fit_rows_saved += len(fit_part)

    if not early_part.empty:
        early_part.to_parquet(
            os.path.join(
                early_stop_folder_112,
                "early_" + filename
            ),
            index=False
        )
        early_stop_rows_saved += len(early_part)

    del part, fit_part, early_part
    gc.collect()

    if i % 10 == 0 or i == len(expanded_files):
        print(f"Processed {i}/{len(expanded_files)} partitions")

print("\nFitting rows saved:", f"{fit_rows_saved:,}")
print("Expected fitting rows:", f"{len(fit_days_112) * 30_490:,}")

print(
    "\nEarly-stopping rows saved:",
    f"{early_stop_rows_saved:,}"
)
print(
    "Expected early-stopping rows:",
    f"{len(early_stop_days_112) * 30_490:,}"
)

print(
    "\nCounts correct:",
    fit_rows_saved == len(fit_days_112) * 30_490
    and early_stop_rows_saved == len(early_stop_days_112) * 30_490
)

Processed 10/30 partitions
Processed 20/30 partitions
Processed 30/30 partitions

Fitting rows saved: 2,713,610
Expected fitting rows: 2,713,610

Early-stopping rows saved: 701,270
Expected early-stopping rows: 701,270

Counts correct: True


In [37]:
import os
import glob
import gc
import pandas as pd
import psutil

fit_files_112 = sorted(
    glob.glob(os.path.join(fit_folder_112, "*.parquet"))
)

# Use the exact feature set from our first model
columns_112 = active_feature_cols + ["sales"]

fit_frames_112 = []

for i, file in enumerate(fit_files_112, start=1):
    part = pd.read_parquet(
        file,
        columns=columns_112
    )

    # Align categorical levels with the original model
    for col in active_categorical_features:
        part[col] = pd.Categorical(
            part[col].astype("string"),
            categories=category_vocab[col]
        )

    fit_frames_112.append(part)

    if i % 10 == 0:
        print(f"Loaded {i}/{len(fit_files_112)} fitting partitions")

# Combine the fitting partitions
fit_df_112 = pd.concat(
    fit_frames_112,
    ignore_index=True,
    copy=False
)

del fit_frames_112
gc.collect()

print("\nExpanded fitting data loaded.")
print("Shape:", fit_df_112.shape)
print("Expected shape:", (len(fit_days_112) * 30_490, 22))

print(
    "Missing targets:",
    fit_df_112["sales"].isna().sum()
)

print(
    f"DataFrame memory: "
    f"{fit_df_112.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

memory = psutil.virtual_memory()
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")

Loaded 10/30 fitting partitions
Loaded 20/30 fitting partitions
Loaded 30/30 fitting partitions

Expanded fitting data loaded.
Shape: (2713610, 22)
Expected shape: (2713610, 22)
Missing targets: 0
DataFrame memory: 121.88 MB
Available RAM: 0.57 GB


In [38]:
import gc
import lightgbm as lgb

# Features and target from the expanded fitting sample
X_fit_112 = fit_df_112[active_feature_cols]
y_fit_112 = fit_df_112["sales"]

# Use the same dataset settings as our first model
dataset_params_112 = {
    "max_bin": 63
}

print("Building expanded LightGBM dataset...")

train_data_112 = lgb.Dataset(
    X_fit_112,
    label=y_fit_112,
    categorical_feature=active_categorical_features,
    params=dataset_params_112,
    free_raw_data=True
)

train_data_112.construct()

print("\nDataset constructed.")
print("Training rows:", train_data_112.num_data())
print("Features:", train_data_112.num_feature())

# Release the raw training DataFrames after construction
del X_fit_112, y_fit_112, fit_df_112
gc.collect()

print("\nRaw fitting DataFrames released.")
print("The expanded Parquet partitions remain saved on disk.")

Building expanded LightGBM dataset...

Dataset constructed.
Training rows: 2713610
Features: 21

Raw fitting DataFrames released.
The expanded Parquet partitions remain saved on disk.


In [39]:
import os
import glob
import gc
import pandas as pd
import lightgbm as lgb

early_files_112 = sorted(
    glob.glob(
        os.path.join(early_stop_folder_112, "*.parquet")
    )
)

early_frames_112 = []

for i, file in enumerate(early_files_112, start=1):
    part = pd.read_parquet(
        file,
        columns=active_feature_cols + ["sales"]
    )

    # Ensure category levels match the training dataset
    for col in active_categorical_features:
        part[col] = pd.Categorical(
            part[col].astype("string"),
            categories=category_vocab[col]
        )

    # Keep numeric feature dtypes consistent
    for col in ["sell_price", "price_change", "price_changed"]:
        part[col] = pd.to_numeric(
            part[col],
            errors="coerce"
        ).astype("float32")

    part["sales"] = part["sales"].astype("float32")
    early_frames_112.append(part)

    if i % 10 == 0:
        print(f"Loaded {i}/{len(early_files_112)} partitions")

early_df_112 = pd.concat(
    early_frames_112,
    ignore_index=True,
    copy=False
)

del early_frames_112
gc.collect()

print("\nEarly-stopping data shape:", early_df_112.shape)
print("Expected rows:", len(early_stop_days_112) * 30_490)
print("Missing targets:", early_df_112["sales"].isna().sum())

# Construct LightGBM validation dataset using training bins
X_es_112 = early_df_112[active_feature_cols]
y_es_112 = early_df_112["sales"]

valid_data_112 = lgb.Dataset(
    X_es_112,
    label=y_es_112,
    categorical_feature=active_categorical_features,
    reference=train_data_112,
    params={"max_bin": 63},
    free_raw_data=True
)

valid_data_112.construct()

print("\nEarly-stopping dataset constructed.")
print("Validation rows:", valid_data_112.num_data())
print("Features:", valid_data_112.num_feature())

# Release raw early-stopping DataFrames
del early_df_112, X_es_112, y_es_112
gc.collect()

print("\nRaw early-stopping DataFrames released.")

Loaded 10/10 partitions

Early-stopping data shape: (701270, 22)
Expected rows: 701270
Missing targets: 0

Early-stopping dataset constructed.
Validation rows: 701270
Features: 21

Raw early-stopping DataFrames released.


In [40]:
import lightgbm as lgb
import gc
import psutil

params_112 = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.1,
    "metric": "rmse",

    # Same model complexity as the original baseline
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 100,
    "max_bin": 63,

    # Memory and compute settings
    "num_threads": 2,
    "force_col_wise": True,

    "seed": 42,
    "verbosity": -1
}

print("Starting expanded LightGBM training...")
print(
    f"Available RAM before training: "
    f"{psutil.virtual_memory().available / (1024**3):.2f} GB"
)

model_112 = lgb.train(
    params=params_112,
    train_set=train_data_112,
    num_boost_round=800,
    valid_sets=[valid_data_112],
    valid_names=["early_stop"],
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=40,
            first_metric_only=True
        ),
        lgb.log_evaluation(period=25)
    ]
)

gc.collect()

print("\nExpanded model training completed.")
print("Best iteration:", model_112.best_iteration)
print(
    "Best internal validation RMSE:",
    model_112.best_score["early_stop"]["rmse"]
)
print(
    f"Available RAM after training: "
    f"{psutil.virtual_memory().available / (1024**3):.2f} GB"
)

Starting expanded LightGBM training...
Available RAM before training: 0.98 GB
Training until validation scores don't improve for 40 rounds
[25]	early_stop's rmse: 2.30544
[50]	early_stop's rmse: 2.05625
[75]	early_stop's rmse: 2.02039
[100]	early_stop's rmse: 2.01164
[125]	early_stop's rmse: 2.0077
[150]	early_stop's rmse: 2.00614
[175]	early_stop's rmse: 2.00516
[200]	early_stop's rmse: 2.00442
[225]	early_stop's rmse: 2.00363
[250]	early_stop's rmse: 2.00395
Early stopping, best iteration is:
[224]	early_stop's rmse: 2.00361
Evaluated only: rmse

Expanded model training completed.
Best iteration: 224
Best internal validation RMSE: 2.0036122731442374
Available RAM after training: 1.01 GB


In [41]:
import numpy as np
import pandas as pd
import gc

# Keep the original model's results unchanged
recursive_predictions_112 = np.zeros(
    (30_490, HORIZON),
    dtype=np.float32
)

# Start with actual history through d_1885
recursive_history_112 = history_sales.copy()

for step, day in enumerate(
    range(VALID_START_D, VALID_END_D + 1)
):
    day_df = validation_work[
        validation_work["d"] == day
    ].copy()

    # Ensure series order matches the historical-sales matrix
    series_codes = day_df["id"].cat.codes.to_numpy(
        dtype=np.int32
    )

    assert len(day_df) == 30_490
    assert np.array_equal(
        series_codes,
        np.arange(30_490)
    ), f"Series order mismatch on day {day}"

    # Calendar, product, store, event and price features
    X_day = day_df[static_feature_cols].copy()

    # Recalculate historical-demand features
    X_day["sales_lag_1"] = recursive_history_112[:, -1]
    X_day["sales_lag_7"] = recursive_history_112[:, -7]
    X_day["sales_lag_28"] = recursive_history_112[:, -28]

    X_day["rolling_mean_7"] = (
        recursive_history_112[:, -7:].mean(axis=1)
    )
    X_day["rolling_mean_28"] = (
        recursive_history_112.mean(axis=1)
    )

    X_day = X_day[active_feature_cols]

    # Predict with the expanded model
    day_predictions = model_112.predict(
        X_day,
        num_iteration=model_112.best_iteration
    ).astype(np.float32)

    recursive_predictions_112[:, step] = day_predictions

    # Feed predictions—not actual future sales—into the next day
    recursive_history_112 = np.concatenate(
        [
            recursive_history_112[:, 1:],
            day_predictions.reshape(-1, 1)
        ],
        axis=1
    )

    if (step + 1) % 7 == 0:
        print(f"Completed {step + 1}/{HORIZON} forecast days")

    del day_df, X_day, day_predictions
    gc.collect()

# Compare all models on exactly the same observations
actual = actual_validation_sales.ravel()

def forecast_metrics(actual_values, predictions):
    errors = np.asarray(predictions).ravel() - actual_values

    return {
        "MAE": np.abs(errors).mean(),
        "RMSE": np.sqrt(np.mean(errors ** 2)),
        "WAPE (%)": (
            np.abs(errors).sum()
            / np.abs(actual_values).sum()
            * 100
        )
    }

comparison = pd.DataFrame({
    "Original model (56 dates)": forecast_metrics(
        actual, recursive_predictions
    ),
    "Expanded model (112 dates)": forecast_metrics(
        actual, recursive_predictions_112
    ),
    "Seasonal naive": forecast_metrics(
        actual, naive_predictions
    )
})

print("\n28-Day Recursive Forecast Comparison:")
print(comparison.round(4))

Completed 7/28 forecast days
Completed 14/28 forecast days
Completed 21/28 forecast days
Completed 28/28 forecast days

28-Day Recursive Forecast Comparison:
          Original model (56 dates)  Expanded model (112 dates)  \
MAE                          1.0661                      1.0887   
RMSE                         2.2090                      2.2027   
WAPE (%)                    76.8946                     78.5227   

          Seasonal naive  
MAE               1.2324  
RMSE              2.7595  
WAPE (%)         88.8867  


In [42]:
import lightgbm as lgb
import gc
import psutil

# Start with the same parameters as the expanded Tweedie model
params_poisson = params_112.copy()

# Change only the objective
params_poisson["objective"] = "poisson"
params_poisson.pop("tweedie_variance_power", None)

print("Training LightGBM Poisson model...")
print(
    f"Available RAM: "
    f"{psutil.virtual_memory().available / (1024**3):.2f} GB"
)

model_poisson_112 = lgb.train(
    params=params_poisson,
    train_set=train_data_112,
    num_boost_round=800,
    valid_sets=[valid_data_112],
    valid_names=["early_stop"],
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=40,
            first_metric_only=True
        ),
        lgb.log_evaluation(period=25)
    ]
)

gc.collect()

print("\nPoisson training completed.")
print("Best iteration:", model_poisson_112.best_iteration)
print(
    "Best internal validation RMSE:",
    model_poisson_112.best_score["early_stop"]["rmse"]
)
print(
    f"Available RAM after training: "
    f"{psutil.virtual_memory().available / (1024**3):.2f} GB"
)

Training LightGBM Poisson model...
Available RAM: 1.04 GB
Training until validation scores don't improve for 40 rounds
[25]	early_stop's rmse: 2.57034
[50]	early_stop's rmse: 2.21867
[75]	early_stop's rmse: 2.10581
[100]	early_stop's rmse: 2.07056
[125]	early_stop's rmse: 2.0543
[150]	early_stop's rmse: 2.04665
[175]	early_stop's rmse: 2.04174
[200]	early_stop's rmse: 2.03907
[225]	early_stop's rmse: 2.03755
[250]	early_stop's rmse: 2.03591
[275]	early_stop's rmse: 2.03506
[300]	early_stop's rmse: 2.03399
[325]	early_stop's rmse: 2.03302
[350]	early_stop's rmse: 2.03225
[375]	early_stop's rmse: 2.03171
[400]	early_stop's rmse: 2.03176
[425]	early_stop's rmse: 2.03098
[450]	early_stop's rmse: 2.0302
[475]	early_stop's rmse: 2.03006
[500]	early_stop's rmse: 2.0299
[525]	early_stop's rmse: 2.03009
Early stopping, best iteration is:
[487]	early_stop's rmse: 2.0298
Evaluated only: rmse

Poisson training completed.
Best iteration: 487
Best internal validation RMSE: 2.0297984566010143
Availab

In [48]:
df=pd.read_parquet("D:\\PIYUSH KUMAR MISHRA\\JOB_PREPARATION\\DataScience\\MachineLearningProjects\\ai-supply-chain-intelligence\\data\\processed\\feature_engineered_chunks\\CA_1_0000.parquet")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 609800 entries, 0 to 609799
Data columns (total 34 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   id               609800 non-null  category      
 1   item_id          609800 non-null  category      
 2   dept_id          609800 non-null  category      
 3   cat_id           609800 non-null  category      
 4   store_id         609800 non-null  category      
 5   state_id         609800 non-null  category      
 6   d                609800 non-null  int16         
 7   sales            609800 non-null  int16         
 8   date             609800 non-null  datetime64[ns]
 9   wm_yr_wk         609800 non-null  int16         
 10  weekday          609800 non-null  category      
 11  wday             609800 non-null  int8          
 12  month            609800 non-null  int8          
 13  year             609800 non-null  int16         
 14  event_name_1     609

In [49]:
import os
import sys
import glob
import gc
import importlib.util
import subprocess

# Install DuckDB if necessary
if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "duckdb"
    ])

import duckdb

# --------------------------------------------------
# 1. Locate the 100 validated source files
# --------------------------------------------------

feature_folder = "../data/processed/feature_engineered_chunks"

feature_files = sorted(
    glob.glob(os.path.join(feature_folder, "*.parquet"))
)

assert len(feature_files) == 100, (
    f"Expected 100 files, found {len(feature_files)}"
)

# Keep the output outside the input folder
output_path = os.path.abspath(
    "../data/processed/feature_engineered_full.parquet"
)

temp_folder = os.path.abspath(
    "../data/processed/duckdb_temp"
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)
os.makedirs(temp_folder, exist_ok=True)

# Do not overwrite the master file without knowing why it exists
if os.path.exists(output_path):
    raise FileExistsError(
        f"Master file already exists: {output_path}\n"
        "Inspect or rename it before rerunning this cell."
    )

# Escape paths for SQL
def sql_quote(value):
    return "'" + value.replace("\\", "/").replace("'", "''") + "'"

file_list_sql = "[" + ", ".join(
    sql_quote(os.path.abspath(path))
    for path in feature_files
) + "]"

# --------------------------------------------------
# 2. Combine using DuckDB
# --------------------------------------------------

con = duckdb.connect()

con.execute("SET memory_limit = '1GB'")
con.execute("SET threads = 2")
con.execute(
    f"SET temp_directory = {sql_quote(temp_folder)}"
)

print("Source files:", len(feature_files))
print("Combining into:", output_path)
print("Expected rows: 58,327,370")

try:
    con.execute(f"""
        COPY (
            SELECT *
            FROM read_parquet(
                {file_list_sql},
                union_by_name = true
            )
        )
        TO {sql_quote(output_path)}
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            ROW_GROUP_SIZE 250000
        )
    """)

    print("\nMaster file written. Validating...")

    # --------------------------------------------------
    # 3. Validate row count and day coverage
    # --------------------------------------------------

    total_rows, min_day, max_day, unique_days = con.execute(f"""
        SELECT
            COUNT(*),
            MIN(d),
            MAX(d),
            COUNT(DISTINCT d)
        FROM read_parquet({sql_quote(output_path)})
    """).fetchone()

    # Check store-level coverage without loading all rows
    store_check = con.execute(f"""
        SELECT
            store_id,
            COUNT(*) AS rows,
            COUNT(DISTINCT item_id) AS items,
            COUNT(DISTINCT d) AS days
        FROM read_parquet({sql_quote(output_path)})
        GROUP BY store_id
        ORDER BY store_id
    """).df()

    print(f"\nTotal rows: {total_rows:,}")
    print(f"Expected rows: {30_490 * 1_913:,}")
    print(f"Day range: d_{min_day} to d_{max_day}")
    print("Unique days:", unique_days)

    print("\nRows per store:")
    print(store_check.to_string(index=False))

    rows_correct = total_rows == 58_327_370
    stores_correct = (
        len(store_check) == 10
        and (store_check["rows"] == 5_832_737).all()
        and (store_check["items"] == 3_049).all()
        and (store_check["days"] == 1_913).all()
    )

    print("\nTotal row count correct:", rows_correct)
    print("Store coverage correct:", stores_correct)

    if not (rows_correct and stores_correct and
            min_day == 1 and max_day == 1913 and unique_days == 1913):
        raise ValueError(
            "Validation failed. Keep the source files unchanged "
            "and inspect the master file before continuing."
        )

    print(
        f"\nMaster file size: "
        f"{os.path.getsize(output_path) / (1024**3):.2f} GB"
    )
    print("MASTER FILE VALIDATED.")

finally:
    con.close()
    gc.collect()

Source files: 100
Combining into: D:\PIYUSH KUMAR MISHRA\JOB_PREPARATION\DataScience\MachineLearningProjects\ai-supply-chain-intelligence\data\processed\feature_engineered_full.parquet
Expected rows: 58,327,370


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Master file written. Validating...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Total rows: 58,327,370
Expected rows: 58,327,370
Day range: d_1 to d_1913
Unique days: 1913

Rows per store:
store_id    rows  items  days
    CA_1 5832737   3049  1913
    CA_2 5832737   3049  1913
    CA_3 5832737   3049  1913
    CA_4 5832737   3049  1913
    TX_1 5832737   3049  1913
    TX_2 5832737   3049  1913
    TX_3 5832737   3049  1913
    WI_1 5832737   3049  1913
    WI_2 5832737   3049  1913
    WI_3 5832737   3049  1913

Total row count correct: True
Store coverage correct: True

Master file size: 0.12 GB
MASTER FILE VALIDATED.


In [50]:
import os
import duckdb

master_path = os.path.abspath(
    "../data/processed/feature_engineered_full.parquet"
)

con = duckdb.connect()

escaped_path = (
    master_path.replace("\\", "/").replace("'", "''")
)

schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{escaped_path}')"
).df()

print("Master file:", master_path)
print("Column count:", len(schema))
print("\nMaster Parquet schema:")
print(
    schema[["column_name", "column_type"]]
    .to_string(index=False)
)

con.close()

Master file: D:\PIYUSH KUMAR MISHRA\JOB_PREPARATION\DataScience\MachineLearningProjects\ai-supply-chain-intelligence\data\processed\feature_engineered_full.parquet
Column count: 34

Master Parquet schema:
    column_name  column_type
             id      VARCHAR
        item_id      VARCHAR
        dept_id      VARCHAR
         cat_id      VARCHAR
       store_id      VARCHAR
       state_id      VARCHAR
              d     SMALLINT
          sales     SMALLINT
           date TIMESTAMP_NS
       wm_yr_wk     SMALLINT
        weekday      VARCHAR
           wday      TINYINT
          month      TINYINT
           year     SMALLINT
   event_name_1      VARCHAR
   event_type_1      VARCHAR
   event_name_2      VARCHAR
   event_type_2      VARCHAR
        snap_CA      TINYINT
        snap_TX      TINYINT
        snap_WI      TINYINT
     sell_price        FLOAT
    day_of_week      TINYINT
     is_weekend      TINYINT
       is_event      TINYINT
    snap_active      TINYINT
price_availa